In [1]:
import time
import numpy as np
from simple_pe.param_est import metric, pe, matched_filter_network
from simple_pe.waveforms import waveform_modes, waveform, parameter_bounds
from scipy import optimize
from pycbc.psd import aLIGOZeroDetHighPower
from pycbc.filter import sigma
from pycbc.noise import frequency_noise_from_psd
from pesummary.utils.array import Array
from pesummary.utils.samples_dict import SamplesDict
from skopt import forest_minimize, gbrt_minimize, gp_minimize
from skopt import Optimizer
from joblib import Parallel, delayed
from functools import partial

import logging
_logger = logging.getLogger('PESummary')
_logger.setLevel(logging.CRITICAL + 10)
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pykerr/qnm.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this p

lal.MSUN_SI != Msun


In [2]:
def _neg_net_snr(
    x, dx_directions, ifos, data, psds, t_start, t_end, f_low, approximant,
    fixed_pars=None, harm2=False, verbose=False
):
    s = dict(zip(dx_directions, x))
    if fixed_pars is not None:
        s.update(fixed_pars)

    if verbose:
        print('making waveform at parameters')
        print(s)
    try:
        h = waveform.make_waveform(
            s, psds[ifos[0]].delta_f, f_low, len(psds[ifos[0]]), approximant,
            harm2=harm2
        )
    except RuntimeError:
        print('error making waveform')
        return np.inf

    s = matched_filter_network(
        ifos, data, psds, t_start, t_end, h, f_low,
        subsample_interpolation=True
    )[0]

    if verbose:
        print('snr = %.4f' % s)

    return -s


def find_peak_snr(
    ifos, data, psds, t_start, t_end, x, dx_directions, f_low,
    approximant="IMRPhenomD", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=None, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
):
    snr_peak = 0

    if method not in ["scipy"]:
        print(
            'Have only implemented scipy'
        )
        return

    elif method == 'scipy':

        nlc = None
        if bounds is None:
            bounds = parameter_bounds.param_bounds(x, dx_directions, harm2)

        nlc = None
        # generate constraint on spins:
        chia = "chi_eff" if "chi_eff" in x.keys() else "chi_align"
        chip = "chi_p2" if "chi_p2" in x.keys() else "chi_p"
        if chip == "chi_p2":
            n = 1
        else:
            n = 2

        cond = (
            (chia in x) and (chip in x) and
            ((chia in dx_directions) or (chip in dx_directions))
        )
        if cond:
            # need bounds based on spin limits
            if (chia in dx_directions) and (chip in dx_directions):
                con = (
                    lambda y: y[dx_directions.index(chia)] ** 2 +
                    y[dx_directions.index(chip)] ** n
                )
                nlc = optimize.NonlinearConstraint(
                    con, parameter_bounds.param_mins['a_1'],
                    parameter_bounds.param_maxs['a_1']
                )

        x0 = np.array([x[k] for k in dx_directions]).flatten()
        fixed_pars = {
            k: float(v) for k, v in x.items() if k not in dx_directions
        }

        if nlc is not None:
            out = optimize.minimize(
                _neg_net_snr, x0, args=(
                    dx_directions, ifos, data, psds, t_start, t_end,
                    f_low, approximant, fixed_pars, harm2, verbose
                ), bounds=bounds, method=scipy_method, options=scipy_opts,
                constraints=nlc
            )

        else:
            out = optimize.minimize(
                _neg_net_snr, x0, args=(
                    dx_directions, ifos, data, psds, t_start, t_end,
                    f_low, approximant, fixed_pars, harm2, verbose
                ), bounds=bounds, method=scipy_method, options=scipy_opts
            )

        x = {}
        for dx, val in zip(dx_directions, out.x):
            x[dx] = val
        x.update(fixed_pars)

        snr_peak = -out.fun


    return x, snr_peak, out['nfev']

In [15]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}

start = time.time()
x, snr_peak, nfev = find_peak_snr(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=None, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

66.05 seconds
SNR: 19.9855, nfev: 184
{'chirp_mass': 23.838068082288512, 'symmetric_mass_ratio': 0.21294408995819689, 'chi_align': 0.011713490534954277}


In [91]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}

start = time.time()
x, snr_peak, nfev = find_peak_snr(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method=None,
    scipy_opts={'eps': [1e-4, 1e-6, 1e-5]}, harm2=False, bounds=None, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

60.41 seconds
SNR: 19.9855, nfev: 188
{'chirp_mass': 23.837946416838104, 'symmetric_mass_ratio': 0.21298376987050754, 'chi_align': 0.011537421173635256}


In [16]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}

start = time.time()
x, snr_peak, nfev = find_peak_snr(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method='Nelder-Mead',
    scipy_opts=None, harm2=False, bounds=None, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

50.96 seconds
SNR: 19.9855, nfev: 142
{'chirp_mass': 23.83841195131967, 'symmetric_mass_ratio': 0.21294018673498352, 'chi_align': 0.01176566183405983}


In [3]:
def find_peak_snr_scikit(
    ifos, data, psds, t_start, t_end, x, dx_directions, f_low,
    approximant="IMRPhenomD", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=None, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
):
    snr_peak = 0

    if method not in ["scipy"]:
        print(
            'Have only implemented scipy'
        )
        return

    elif method == 'scipy':

        nlc = None
        if bounds is None:
            bounds = parameter_bounds.param_bounds(x, dx_directions, harm2)

        nlc = None
        # generate constraint on spins:
        chia = "chi_eff" if "chi_eff" in x.keys() else "chi_align"
        chip = "chi_p2" if "chi_p2" in x.keys() else "chi_p"
        if chip == "chi_p2":
            n = 1
        else:
            n = 2

        cond = (
            (chia in x) and (chip in x) and
            ((chia in dx_directions) or (chip in dx_directions))
        )
        if cond:
            # need bounds based on spin limits
            if (chia in dx_directions) and (chip in dx_directions):
                con = (
                    lambda y: y[dx_directions.index(chia)] ** 2 +
                    y[dx_directions.index(chip)] ** n
                )
                nlc = optimize.NonlinearConstraint(
                    con, parameter_bounds.param_mins['a_1'],
                    parameter_bounds.param_maxs['a_1']
                )

        x0 = np.array([x[k] for k in dx_directions]).flatten()
        fixed_pars = {
            k: float(v) for k, v in x.items() if k not in dx_directions
        }

        if nlc is not None:
            out = optimize.minimize(
                _neg_net_snr, x0, args=(
                    dx_directions, ifos, data, psds, t_start, t_end,
                    f_low, approximant, fixed_pars, harm2, verbose
                ), bounds=bounds, method=scipy_method, options=scipy_opts,
                constraints=nlc
            )

        else:
            _neg_net_snr_part = partial(_neg_net_snr, dx_directions=dx_directions, ifos=ifos, data=data, psds=psds, t_start=t_start, t_end=t_end,
                    f_low=f_low, approximant=approximant, fixed_pars=fixed_pars, harm2=harm2, verbose=verbose)
            # optimizer = Optimizer(
            #     dimensions=bounds,
            #     random_state=1234,
            #     base_estimator='gp',
            #     acq_func='EI',
            #     n_random_starts=40
            # )
            # with Parallel(n_jobs=4, prefer="threads") as parallel:
            #     for i in range(20):
            #         x = optimizer.ask(n_points=4)  # x is a list of n_points points
            #         y = parallel(delayed(_neg_net_snr_part)(v) for v in x)  # evaluate points in parallel
            #         optimizer.tell(x, y)
            # minx = optimizer.Xi[np.argmin(optimizer.yi)]
            # miny = np.min(optimizer.yi)
            out = gp_minimize(
                _neg_net_snr_part,                  # the function to minimize
                bounds,      # the bounds on each dimension of x
                acq_func="EI",      # the acquisition function
                n_calls=200,         # the number of evaluations of f
                n_random_starts=50,  # the number of random initialization points
                random_state=1234
            )
            # out = optimize.minimize(
            #     _neg_net_snr, x0, args=(
            #         dx_directions, ifos, data, psds, t_start, t_end,
            #         f_low, approximant, fixed_pars, harm2, verbose
            #     ), bounds=bounds, method=scipy_method, options=scipy_opts
            # )

        x = {}
        for dx, val in zip(dx_directions, out.x):
            x[dx] = val
        x.update(fixed_pars)

        snr_peak = -out.fun

    return x, snr_peak, len(out.func_vals)

In [32]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24.0, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}
bounds = [(23.5, 24.5), (0.21, 0.23), (-0.1, 0.1)]

start = time.time()
x, snr_peak, nfev = find_peak_snr_scikit(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

60.72 seconds
SNR: 19.9817, nfev: 60
{'chirp_mass': 23.845122315280417, 'symmetric_mass_ratio': 0.21486767851030833, 'chi_align': 0.0023225676545286095}


In [34]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24.0, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}
bounds = [(23.5, 24.5), (0.21, 0.23), (-0.1, 0.1)]

start = time.time()
x, snr_peak, nfev = find_peak_snr_scikit(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

168.15 seconds
SNR: 19.9831, nfev: 120
{'chirp_mass': 23.796014888515316, 'symmetric_mass_ratio': 0.21276192553718987, 'chi_align': 0.006523879394415719}


Adding gaussian noise:

In [37]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
data['H1'] += frequency_noise_from_psd(psds['H1'], seed=5678)
init_guess = {'chirp_mass': 24.4, 'symmetric_mass_ratio': 0.215, 'chi_align': 0.1}
bounds = [(23.5, 24.5), (0.21, 0.23), (-0.1, 0.1)]

start = time.time()
x, snr_peak, nfev = find_peak_snr(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method='Nelder-Mead',
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

36.27 seconds
SNR: 20.6359, nfev: 118
{'chirp_mass': 23.744557973469696, 'symmetric_mass_ratio': 0.21000000000000052, 'chi_align': 0.00806769377268235}


In [9]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
data['H1'] += frequency_noise_from_psd(psds['H1'], seed=5678)
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}
bounds = [(23.5, 24.5), (0.21, 0.23), (-0.1, 0.1)]

start = time.time()
x, snr_peak, nfev = find_peak_snr_scikit(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

69.42 seconds
SNR: 20.6342, nfev: 60
{'chirp_mass': 23.75371335979321, 'symmetric_mass_ratio': 0.21, 'chi_align': 0.007705515165253904}


In [29]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
data['H1'] += frequency_noise_from_psd(psds['H1'], seed=5678)
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}
bounds = [(23.5, 24.5), (0.21, 0.23), (-0.1, 0.1)]

start = time.time()
x, snr_peak, nfev = find_peak_snr_scikit(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

87.46 seconds
SNR: 20.6250, nfev: 200
{'chirp_mass': 23.640299507527423, 'symmetric_mass_ratio': 0.2108660848356135, 'chi_align': -0.007974163370591744}


In [27]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24, 'symmetric_mass_ratio': 2/9, 'chi_align': 0}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'IMRPhenomXPHM')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
data['H1'] += frequency_noise_from_psd(psds['H1'], seed=5678)
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1}
bounds = [(23.5, 24.5), (0.21, 0.23), (-0.1, 0.1)]

start = time.time()
x, snr_peak, nfev = find_peak_snr_scikit(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="IMRPhenomXPHM", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

5.80 seconds
SNR: 20.0653, nfev: 15
{'chirp_mass': 24.183462935172138, 'symmetric_mass_ratio': 0.224254040539658, 'chi_align': -0.025949849041921005}


What about including eccentricity

In [42]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24.0, 'symmetric_mass_ratio': 2/9, 'chi_align': 0, 'ecc10sqrd': 0.2**2}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'TEOBResumS-Dali')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1, 'ecc10sqrd': 0.1**2}
bounds = [(20.0, 30.0), (0.18, 0.24), (-0.3, 0.3), (0.0, 0.5**2)]

start = time.time()
x, snr_peak, nfev = find_peak_snr_scikit(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="TEOBResumS-Dali", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)

104.44 seconds
SNR: 19.0037, nfev: 60
{'chirp_mass': 23.866726909004413, 'symmetric_mass_ratio': 0.24, 'chi_align': -0.12430201399143234, 'ecc10sqrd': 0.0}


In [ ]:
ifos = ['H1']
psds = {'H1': aLIGOZeroDetHighPower(int(32*4096)//2+1, 1/32, 20)}
params = {'chirp_mass': 24.0, 'symmetric_mass_ratio': 2/9, 'chi_align': 0, 'ecc10sqrd': 0.2**2}
data = {'H1': waveform.make_waveform(params, psds[ifos[0]].delta_f, 20, len(psds[ifos[0]]), 'TEOBResumS-Dali')}
raw_snr = sigma(data['H1'], psds['H1'], low_frequency_cutoff=20)
data['H1'] = data['H1'] * 20/raw_snr
init_guess = {'chirp_mass': 25, 'symmetric_mass_ratio': 0.2, 'chi_align': 0.1, 'ecc10sqrd': 0.1**2}
bounds = [(20.0, 30.0), (0.18, 0.24), (-0.3, 0.3), (0.0, 0.5**2)]

start = time.time()
x, snr_peak, nfev = find_peak_snr_scikit(
    ifos, data, psds, -0.1, 0.1, init_guess, init_guess.keys(), 20,
    approximant="TEOBResumS-Dali", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=bounds, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None
)
end = time.time()
print(f'{end-start:.2f} seconds')
print(f'SNR: {snr_peak:.4f}, nfev: {nfev}')
print(x)